# Trading Strategy Analysis - Interactive Example

This notebook demonstrates how to use the trading analysis library interactively.

## Setup

Import the library and configure display settings.

In [ ]:
# Imports
from analysis import TradingAnalyzer, PerformanceMetrics, PerformanceVisualizer, ReportGenerator
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
%matplotlib inline

## Load Trading Decisions

Choose a CSV file with trading decisions.

In [ ]:
# Specify your CSV file
csv_file = 'results_bulk/AAPL_decisions_2020-01-06_2024-12-23.csv'

# Or list available files
available_files = list(Path('results_bulk').glob('*_decisions_*.csv'))
print(f"Available files: {len(available_files)}")
for f in available_files[:5]:
    print(f"  - {f.name}")

## Initialize Analyzer

In [ ]:
# Create analyzer
analyzer = TradingAnalyzer(
    csv_path=csv_file,
    initial_capital=10000.0,  # $10,000 starting capital
    risk_free_rate=0.03       # 3% annual risk-free rate
)

print(f"Analyzing: {analyzer.ticker}")
print(f"Period: {analyzer.start_date} to {analyzer.end_date}")
print(f"Trading days: {len(analyzer.portfolio_df):,}")

## Calculate Performance Metrics

In [ ]:
# Initialize metrics calculator
metrics = PerformanceMetrics(analyzer)

# Calculate all metrics
all_metrics = metrics.calculate_all_metrics()

# Display as DataFrame
metrics_df = pd.DataFrame([
    {'Metric': k.replace('_', ' ').title(), 'Value': f"{v:.2f}"}
    for k, v in all_metrics.items()
])

print("\n" + "="*60)
print("PERFORMANCE METRICS")
print("="*60)
display(metrics_df)

## Key Metrics Summary

In [ ]:
# Print formatted summary
print(metrics.get_metrics_summary())

## Visualizations

### Equity Curve

In [ ]:
# Create visualizer
visualizer = PerformanceVisualizer(analyzer, metrics)

# Plot equity curve
visualizer.plot_equity_curve(show=True)

### Drawdown Chart

In [ ]:
visualizer.plot_drawdown(show=True)

### Monthly Returns Heatmap

In [ ]:
visualizer.plot_monthly_returns(show=True)

### Trade Distribution

In [ ]:
visualizer.plot_trade_distribution(show=True)

### Returns Distribution

In [ ]:
visualizer.plot_returns_distribution(show=True)

### Rolling Metrics

In [ ]:
visualizer.plot_rolling_metrics(window=252, show=True)  # 252-day (1 year) rolling window

## Trade Log Analysis

In [ ]:
# Get trade log
trade_log = analyzer.get_trade_log()

print(f"Total trades: {len(trade_log)}")
print(f"Winning trades: {trade_log['win'].sum()}")
print(f"Losing trades: {(~trade_log['win']).sum()}")

# Display first few trades
print("\nFirst 10 trades:")
display(trade_log.head(10))

## Portfolio History

In [ ]:
# Display portfolio history
portfolio = analyzer.portfolio_df[['price', 'decision', 'portfolio_value', 'bnh_value', 'drawdown']]

print("Portfolio history (last 10 days):")
display(portfolio.tail(10))

## Generate Complete Report

In [ ]:
# Generate full report with all files
report = ReportGenerator(analyzer, metrics, visualizer)

output_dir = f'reports/{analyzer.ticker}'
saved_files = report.generate_full_report(
    output_dir=output_dir,
    show_charts=False  # Set to True to display charts during generation
)

print(f"\nReport saved to: {output_dir}")
print("\nSaved files:")
for key, value in saved_files.items():
    if key != 'charts':
        print(f"  - {key}: {Path(value).name}")

## Compare Multiple Strategies

In [ ]:
# Analyze multiple files and compare
results = []

for csv_file in available_files[:5]:  # Analyze first 5 files
    try:
        a = TradingAnalyzer(str(csv_file))
        m = PerformanceMetrics(a)
        
        results.append({
            'Ticker': a.ticker,
            'Annual Return (%)': m.annualized_return(),
            'Sharpe Ratio': m.annualized_sharpe_ratio(),
            'Max Drawdown (%)': m.maximum_drawdown(),
            'Win Rate (%)': m.win_rate(),
            'Calmar Ratio': m.calmar_ratio(),
            'Total Trades': m.total_trades()
        })
    except Exception as e:
        print(f"Error with {csv_file.name}: {e}")

# Create comparison DataFrame
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values('Sharpe Ratio', ascending=False)

print("\n" + "="*80)
print("STRATEGY COMPARISON")
print("="*80)
display(comparison_df)

## Export Results

In [ ]:
# Export comparison to CSV
comparison_df.to_csv('strategy_comparison.csv', index=False)
print("Comparison saved to strategy_comparison.csv")

# Export metrics to JSON
import json

with open(f'{analyzer.ticker}_metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

print(f"Metrics saved to {analyzer.ticker}_metrics.json")